[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ana123-hub/decoder-transformer-lab/blob/main/notebooks/02_phase2_tinystories.ipynb)

##Mount drive to save model checkpoints and not loose progress

In [4]:
import os
from google.colab import drive

print("--> Mounting Google Drive...")
# Force remount to clear stale sessions
drive.mount('/content/drive', force_remount=True)

# Create checkpoint directory in Drive
DRIVE_DIR = '/content/drive/MyDrive/decoder_transformer_tinystories'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"--> Checkpoints directory ready at: {DRIVE_DIR}")

# Install dependencies
!pip install -q datasets tokenizers

--> Mounting Google Drive...
Mounted at /content/drive
--> Checkpoints directory ready at: /content/drive/MyDrive/decoder_transformer_tinystories


##Load dataset from Hugging Face and build BPE tokenizer

In [13]:
import os
from datasets import load_dataset
from tokenizers import Tokenizer, ByteLevelBPETokenizer

print("--> Downloading TinyStories dataset from Hugging Face...")
raw_dataset = load_dataset("roneneldan/TinyStories", split="train")
print(f"--> Dataset loaded! Total samples: {len(raw_dataset):,}")

TOKENIZER_PATH = os.path.join(DRIVE_DIR, "tokenizer.json")
VOCAB_SIZE = 4096  # Adjust as needed (e.g., 1000 or 4096)

if os.path.exists(TOKENIZER_PATH):
    print(f"--> Loading tokenizer from Drive: {TOKENIZER_PATH}")
    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
else:
    print("--> Training new BPE Tokenizer on 50k subset...")
    sample_texts = [raw_dataset[i]["text"] for i in range(min(50000, len(raw_dataset)))]

    with open("temp_corpus.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(sample_texts))

    bpe_tokenizer = ByteLevelBPETokenizer()
    bpe_tokenizer.train(
        files=["temp_corpus.txt"],
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=["<pad>", "<unk>", "<s>", "</s>"]
    )
    bpe_tokenizer.save(TOKENIZER_PATH)
    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
    if os.path.exists("temp_corpus.txt"):
        os.remove("temp_corpus.txt")
    print(f"--> Saved tokenizer to Drive: {TOKENIZER_PATH}")

val_raw_dataset = load_dataset("roneneldan/TinyStories", split="validation")
vocab_size = tokenizer.get_vocab_size()
print(f"--> Active Vocabulary Size: {vocab_size}")

--> Downloading TinyStories dataset from Hugging Face...
--> Dataset loaded! Total samples: 2,119,719
--> Loading tokenizer from Drive: /content/drive/MyDrive/decoder_transformer_tinystories/tokenizer.json
--> Active Vocabulary Size: 4096


##Pytorch dataset and DataLoader setup


In [14]:
import torch
from torch.utils.data import Dataset, DataLoader

# Configuration
BLOCK_SIZE = 256    # Context length (sequence length)
BATCH_SIZE = 64
class TinyStoriesDataset(Dataset):
    def __init__(self, dataset, tokenizer, block_size=256):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.block_size = block_size
        self.pad_id = tokenizer.token_to_id("<pad>") if tokenizer.token_to_id("<pad>") is not None else 0

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        text = self.dataset[idx]["text"]
        # Encode text to token IDs
        tokens = self.tokenizer.encode(text).ids

        # Ensure minimum sequence length to avoid empty slices
        if len(tokens) < 2:
            tokens = [self.pad_id] * 2

        # Truncate to block_size + 1 (for input x and target y)
        if len(tokens) > self.block_size + 1:
            tokens = tokens[: self.block_size + 1]
        else:
            # Pad sequence if shorter than block_size + 1
            padding_len = (self.block_size + 1) - len(tokens)
            tokens = tokens + [self.pad_id] * padding_len

        # x: input tokens, y: target next-tokens (shifted by 1)
        x = torch.tensor(tokens[:-1], dtype=torch.long)
        y = torch.tensor(tokens[1:], dtype=torch.long)
        return x, y

val_dataset = TinyStoriesDataset(val_raw_dataset, tokenizer, block_size=BLOCK_SIZE)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("--> Constructing PyTorch Dataset...")
train_dataset = TinyStoriesDataset(raw_dataset, tokenizer, block_size=BLOCK_SIZE)

print("--> Initializing DataLoader...")
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

# Test a single batch throughput
x_sample, y_sample = next(iter(train_loader))
print(f"--> Batch sample successfully generated!")
print(f"    Input Batch Tensor Shape (x):  {x_sample.shape}")   # [64, 256]
print(f"    Target Batch Tensor Shape (y): {y_sample.shape}")  # [64, 256]

--> Constructing PyTorch Dataset...
--> Initializing DataLoader...
--> Batch sample successfully generated!
    Input Batch Tensor Shape (x):  torch.Size([64, 256])
    Target Batch Tensor Shape (y): torch.Size([64, 256])


##Define Transformer Architecture
RMSNorm for pre-normalization stability

Bias-free linear layers (bias=False) in c_attn and c_proj

Weight Tying between token embeddings (wte) and the output head (lm_head)

Flash Attention acceleration via torch.nn.functional.scaled_dot_product_attention

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Model Configuration
class ModelConfig:
    def __init__(self, vocab_size=vocab_size, block_size=BLOCK_SIZE):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_embd = 384
        self.n_head = 6
        self.n_layer = 6
        self.dropout = 0.1

# 2. RMSNorm Layer
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * norm * self.weight

# 3. Causal Self-Attention Layer
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size()

        # Calculate Query, Key, Value vectors
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        # Reshape for multi-head attention: (B, n_head, T, head_size)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        # PyTorch SDPA (uses Causal FlashAttention under the hood on GPU)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        # Re-assemble head outputs
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

# 4. Feed-Forward Network (MLP)
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

# ------------------------------------------------------------------------------
# 5. Transformer Block
# ------------------------------------------------------------------------------
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = RMSNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

# 6. Complete Decoder-Only Transformer
class DecoderTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)]),
            ln_f = RMSNorm(config.n_embd)
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Weight tying: share embedding weights with output projection
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        x = self.transformer.wte(idx)
        x = self.transformer.drop(x)

        for block in self.transformer.h:
            x = block(x)

        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)

        return logits, loss

# Initialize Model instance
model_config = ModelConfig(vocab_size=vocab_size, block_size=BLOCK_SIZE)
model = DecoderTransformer(model_config).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"--> DecoderTransformer Model successfully instantiated on {device}!")
print(f"    Architecture: {model_config.n_layer} layers | {model_config.n_head} heads | {model_config.n_embd} hidden dim")
print(f"    Total Trainable Parameters: {num_params / 1e6:.2f}M")

--> DecoderTransformer Model successfully instantiated on cuda!
    Architecture: 6 layers | 6 heads | 384 hidden dim
    Total Trainable Parameters: 12.19M


##Training Loop with AMP Mixed Precision, Cosine LR Scheduler, Drive Checkpointing and word embedding in model's Forward() pass.
Automatic Mixed Precision (bfloat16/float16) via torch.amp for fast matrix multiplication on Colab's T4/A100 GPUs.

AdamW Optimizer with weight decay to prevent overfitting.

Cosine Learning Rate Decay with a 200-step warmup.

Live Story Generation Callback every 250 steps so you can watch text quality improve in real time.

Automatic Drive Checkpointing to save best_model.pt directly to your Google Drive folder.


In [ ]:
import os
import math
import time
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler

# 1. Hyperparameters & Training Setup
MAX_STEPS = 5000          # Total training steps (~35-45 minutes on T4 GPU)
EVAL_INTERVAL = 250       # Evaluate validation loss & save checkpoint every 250 steps
EVAL_BATCHES = 50         # Number of validation batches to average over
LOG_EVERY = 25            # Log training loss to console every 25 steps
LEARNING_RATE = 5e-4      # Peak learning rate
MIN_LR = 5e-5             # Minimum learning rate after cosine decay
WARMUP_STEPS = 200        # Linear warmup steps
WEIGHT_DECAY = 0.1        # Weight decay for regularization
GRAD_CLIP = 1.0           # Max norm for gradient clipping
GRAD_ACCUM_STEPS = 2      # Effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS (64 * 2 = 128)

# Optimizer configuration: separate 2D params (weights) from 1D params (biases/norms)
decay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() >= 2]
nodecay_params = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() < 2]

optim_groups = [
    {'params': decay_params, 'weight_decay': WEIGHT_DECAY},
    {'params': nodecay_params, 'weight_decay': 0.0}
]

optimizer = torch.optim.AdamW(optim_groups, lr=LEARNING_RATE, betas=(0.9, 0.95), eps=1e-8)

# Cosine Learning Rate Schedule with Linear Warmup
def get_lr(it):
    if it < WARMUP_STEPS:
        return LEARNING_RATE * (it + 1) / WARMUP_STEPS
    if it > MAX_STEPS:
        return MIN_LR
    decay_ratio = (it - WARMUP_STEPS) / (MAX_STEPS - WARMUP_STEPS)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return MIN_LR + coeff * (LEARNING_RATE - MIN_LR)

# Automatic Mixed Precision Setup
is_cuda = device.type == "cuda"
amp_dtype = torch.bfloat16 if (is_cuda and torch.cuda.is_bf16_supported()) else torch.float16
scaler = GradScaler("cuda", enabled=(is_cuda and amp_dtype == torch.float16))

print(f"--> Optimizer initialized with Weight Decay={WEIGHT_DECAY}")
print(f"--> Training for {MAX_STEPS} steps | Mixed Precision: {amp_dtype}")

# 2. Validation Loss Estimator & Story Generation Callbacks
@torch.no_grad()
def estimate_val_loss(model, val_loader, eval_batches=EVAL_BATCHES):
    model.eval()
    total_val_loss = 0.0
    val_iter = iter(val_loader)
    actual_batches = 0

    for _ in range(eval_batches):
        try:
            x_val, y_val = next(val_iter)
        except StopIteration:
            val_iter = iter(val_loader)
            try:
                x_val, y_val = next(val_iter)
            except StopIteration:
                break

        x_val, y_val = x_val.to(device, non_blocking=True), y_val.to(device, non_blocking=True)
        with autocast(device_type=device.type, dtype=amp_dtype, enabled=is_cuda):
            _, loss = model(x_val, targets=y_val)

        total_val_loss += loss.item()
        actual_batches += 1

    avg_val_loss = total_val_loss / max(actual_batches, 1)
    model.train()
    return avg_val_loss


def generate_sample(model, tokenizer, prompt="Once upon a time", max_new_tokens=40):
    model.eval()
    encoded = tokenizer.encode(prompt).ids
    idx = torch.tensor([encoded], dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -BLOCK_SIZE:]
            logits, _ = model(idx_cond)
            logits = logits[:, -1, :] / 0.7  # Temperature = 0.7

            # Top-k sampling (k=20)
            v, _ = torch.topk(logits, min(20, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')
            probs = torch.nn.functional.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)

    decoded = tokenizer.decode(idx[0].tolist())
    model.train()
    return decoded

# 3. Main Training Loop
print("\n🚀 Starting Training Loop...")
model.train()

step = 0
train_iter = iter(train_loader)
start_time = time.time()
best_val_loss = float('inf')

while step < MAX_STEPS:
    # Update Learning Rate according to schedule
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0

    # Gradient Accumulation Loop
    for _ in range(GRAD_ACCUM_STEPS):
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        # Forward Pass with AMP
        with autocast(device_type=device.type, dtype=amp_dtype, enabled=is_cuda):
            _, loss = model(x, targets=y)
            loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        accum_loss += loss.item() * GRAD_ACCUM_STEPS

    # Unscale & Clip Gradients
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

    # Optimizer Step
    scaler.step(optimizer)
    scaler.update()

    step += 1

    # Periodic Training Console Logging
    if step % LOG_EVERY == 0 or step == 1:
        elapsed = time.time() - start_time
        tokens_processed = LOG_EVERY * BATCH_SIZE * BLOCK_SIZE * GRAD_ACCUM_STEPS
        tok_per_sec = tokens_processed / max(elapsed, 1e-5)
        print(f"Step {step:04d}/{MAX_STEPS} | Train Loss: {accum_loss:.4f} | LR: {lr:.2e} | Speed: {tok_per_sec:.0f} tok/s")
        start_time = time.time()

    # Periodic Validation, Story Generation & Checkpoint Saving
    if step % EVAL_INTERVAL == 0 or step == MAX_STEPS:
        print(f"\n--- [ Validation & Evaluation @ Step {step} ] ---")
        val_loss = estimate_val_loss(model, val_loader)
        sample_text = generate_sample(model, tokenizer, prompt="Once upon a time")

        print(f"📊 Train Loss: {accum_loss:.4f} | Val Loss: {val_loss:.4f}")
        print(f"📖 Sample Output:\n{sample_text}\n" + "-"*50)

        # Build Checkpoint Dictionary
        checkpoint_dict = {
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': accum_loss,
            'val_loss': val_loss,
            'config': model_config
        }

        # Save Step Checkpoint
        ckpt_path = os.path.join(DRIVE_DIR, f"checkpoint_step_{step}.pt")
        torch.save(checkpoint_dict, ckpt_path)
        print(f"--> Saved step checkpoint: {ckpt_path}")

        # Check for Best Model based on Validation Loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(DRIVE_DIR, "best_model2.pt")
            torch.save(checkpoint_dict, best_path)
            print(f"🏆 New best validation loss ({best_val_loss:.4f})! Saved to {best_path}")
        print()

print("\n🎉 Phase 2 Training Complete!")

--> Optimizer initialized with Weight Decay=0.1
--> Training for 5000 steps | Mixed Precision: torch.bfloat16

🚀 Starting Training Loop...
Step 0001/5000 | Train Loss: 3.6825 | LR: 2.50e-06 | Speed: 652913 tok/s
Step 0025/5000 | Train Loss: 3.7848 | LR: 6.25e-05 | Speed: 30420 tok/s
Step 0050/5000 | Train Loss: 3.6049 | LR: 1.25e-04 | Speed: 28464 tok/s
Step 0075/5000 | Train Loss: 3.6067 | LR: 1.88e-04 | Speed: 27824 tok/s
Step 0100/5000 | Train Loss: 3.5095 | LR: 2.50e-04 | Speed: 27184 tok/s
Step 0125/5000 | Train Loss: 3.6268 | LR: 3.13e-04 | Speed: 26958 tok/s
Step 0150/5000 | Train Loss: 3.7298 | LR: 3.75e-04 | Speed: 26235 tok/s
Step 0175/5000 | Train Loss: 3.5035 | LR: 4.38e-04 | Speed: 26396 tok/s
Step 0200/5000 | Train Loss: 3.4883 | LR: 5.00e-04 | Speed: 26364 tok/s
Step 0225/5000 | Train Loss: 3.5386 | LR: 5.00e-04 | Speed: 26347 tok/s
Step 0250/5000 | Train Loss: 3.5379 | LR: 5.00e-04 | Speed: 26434 tok/s

--- [ Validation & Evaluation @ Step 250 ] ---
📊 Train Loss: 3.5379